# Rebinning demo: load, cache, bin, inspect, save (globals/config version)

This is the same walkthrough as `session_demo.ipynb`, but written against
`event_processing.rebinning_api.server`'s module-level globals and helper
functions directly, instead of `session.Session` / `event_store.EventStore`
(those live as uncommitted work-in-progress and were dropped from this
branch to merge in other changes).

This matches how `server.py`'s CLI/FastAPI routes actually work: one
process-wide cache configuration (`server.configure_data_source(...)`,
which sets `CACHE_ROOT`/`AUTO_HST_FILE`/`REFRESH_CACHE` and configures
`data_cache`/`event_cache` to match), then `nexus_util.open_nexus_entry`,
`server.ensure_cleaned_events`, `binning.bin`, and `nexus_util.nexus_dup`
called directly.

Steps:
1. Configure the data source and open the nexus entry for one measurement.
2. Load the cleaned events (fetched + disk-cached on first use).
3. Rebin against an arbitrary, user-chosen set of time bin edges.
4. Inspect the binned result.
5. Save the binned result to disk.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
from pathlib import Path

import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from event_processing.rebinning_api import server, models, nexus_util, binning

## 1. Configure the data source and open the nexus entry

`server.configure_data_source(cache=...)` sets the process-wide `CACHE_ROOT`
global (nexus files, and cleaned events once computed) and propagates it to
`data_cache`/`event_cache`. It's a single global configuration for the whole
process -- unlike `EventStore`, you can't have two independent configs at
once in the same kernel.

In [ ]:
FILENAME = "sans74909.nxs.ngv"
CACHE = "../cache"  # repo-root ./cache, relative to notebooks/

server.configure_data_source(cache=CACHE)

measurement = models.Measurement(filename=FILENAME, entry=0, point=0)
entry = nexus_util.open_nexus_entry(measurement, refresh=False)
entry

## 2. Load (and cache) the events

`server.ensure_cleaned_events` is the same helper `bin_events` uses
internally: it loads previously-persisted cleaned events from
`<CACHE_ROOT>/events_cache/<filename>` if present, or else fetches raw
events and cleans them (time-of-flight correction, sorting, etc.) and
persists the result for next time.

In [ ]:
events = server.ensure_cleaned_events(measurement, entry)
list(events)

In [ ]:
print("detectors:", list(events["detectors"]))
# print("devices:  ", list(events["devices"]))
print("T0 triggers:", 0 if events.get("T0") is None else len(events["T0"]))

for name, det in events["detectors"].items():
    print(f"  {name:15s} dims={det['dims']} events={len(det['ts'])}")

## 3. Rebin against arbitrary bin edges

`binning.bin` takes any user-provided set of edges (in seconds) wrapped in a
`models.TimeBins` -- no need to go through the time_linbins/strobe_linbins
helpers unless you want evenly spaced bins.

In [ ]:
# Duration from the last event across all detectors with any counts.
duration_s = max(
    det["ts"].max() for det in events["detectors"].values() if len(det["ts"])
) / 1e9
print(f"measurement duration: {duration_s:.1f} s")

edges = np.linspace(0, duration_s, 1001)  # 1000 arbitrary time bins
bins = models.TimeBins(edges=edges)
result = binning.bin(entry, measurement.point, bins, events)
list(result)

## 4. Inspect the binned data

In [ ]:
for name, data in result["detectors"].items():
    print(f"{name:15s} shape={data.shape} total_counts={data.sum()}")

print("count_time (s) per bin:", result["count_time"])

In [ ]:
# Pick the detector with the most counts to visualize.
detector = max(result["detectors"], key=lambda k: result["detectors"][k].sum())
data = result["detectors"][detector]  # shape: (n_bins, nx, ny)

fig_frame = px.imshow(
    np.log10(data[0].T + 0.5),
    origin="lower",
    color_continuous_scale="Viridis",
    aspect="equal",
    labels={"color": "log10(counts)"},
    title=f"{detector}, bin 0",
)
fig_frame.update_layout(width=500, height=500)
fig_frame.show()

time_trace = data.reshape(data.shape[0], -1).sum(axis=1)
bin_centers = (edges[:-1] + edges[1:]) / 2
fig_trace = go.Figure(
    go.Scatter(x=bin_centers, y=time_trace, mode="lines+markers")
)
fig_trace.update_layout(
    title=f"{detector}, counts vs time",
    xaxis_title="time (s)",
    yaxis_title="counts",
)
fig_trace.show()

## 5. Save the binned data

The binned result is plain numpy arrays, so it saves directly with
`np.savez` -- no need to go through the NeXus-writing path used by the
`event-processing rebin` CLI/web download unless you specifically want a
rebinned NeXus file (see `nexus_util.nexus_dup`, used internally by
`server.get_nexus`).

In [ ]:
out_path = "sans73576_binned.npz"
np.savez(
    out_path,
    edges=edges,
    count_time=result["count_time"],
    **result["detectors"],
)

# Round-trip check
loaded = np.load(out_path)
print(f"saved to {out_path}, keys:", list(loaded.keys()))

## 6. Save as a NeXus file

Instead of (or alongside) the plain-numpy save above, `nexus_util.nexus_dup`
writes the binned result into a copy of the original NeXus file -- the same
thing the `event-processing rebin` CLI command and the web GUI's NeXus
download produce (`server.get_nexus` just calls this and streams the
bytes back instead of writing them to disk). Pass `split=True` to get a ZIP
archive with one NeXus file per bin instead of a single file with all bins.

In [ ]:
data, outfile, mimetype = nexus_util.nexus_dup(entry, result, bins, filename=measurement.filename)
nexus_path = Path(outfile)
nexus_path.write_bytes(data)
print(f"wrote {nexus_path} ({len(data)} bytes)")

zip_data, zip_outfile, zip_mimetype = nexus_util.nexus_dup(entry, result, bins, split=True, filename=measurement.filename)
zip_path = Path(zip_outfile)
zip_path.write_bytes(zip_data)
print(f"wrote {zip_path} ({len(zip_data)} bytes)")

In [ ]:
entry.file.close()  # release the underlying h5py file handle